# T04: Schema Classes and Element Mappings

This tutorial demonstrates cross-source element mappings and curated alias groups.
You will create a mapping between two elements from different sources, create an
alias group, and run the `detect-aliases` CLI command.

**Services required**: backend (`http://localhost:8002`)

**Est. time**: 10 min

In [1]:
# Cell 2 — service availability check + fetch two elements for use as mapping targets
import os
import subprocess
from pathlib import Path

import httpx

BACKEND_URL = os.getenv("BACKEND_URL", "http://localhost:8002")
API_KEY = os.getenv(
    "API_KEY",
    "qs005testtoken1234567890abcdef1234567890abcdef1234567890abcdef12",
)
HEADERS = {"Authorization": f"Bearer {API_KEY}"}
INGESTION_DIR = os.getenv(
    "INGESTION_DIR",
    str(Path("../ingestion").resolve()),
)

try:
    httpx.get(f"{BACKEND_URL}/health", timeout=2.0).raise_for_status()
    print(f"✓ Backend available at {BACKEND_URL}")
except Exception as _e:
    import pytest

    pytest.skip(f"Backend unavailable: {_e}")

# Fetch elements with no existing mappings to use as mapping source and target
_resp = httpx.get(
    f"{BACKEND_URL}/api/v1/elements/",
    headers=HEADERS,
    params={"limit": 50},
    timeout=5.0,
)
assert _resp.status_code == 200
_items = [it for it in _resp.json()["items"] if it.get("mapping_count", 0) == 0]

if len(_items) < 2:
    import pytest

    pytest.skip("Fewer than 2 unmapped elements found — run T02 (02_ingest_schemas.ipynb) first")

elem_a_id = _items[0]["id"]
elem_b_id = _items[1]["id"]
print(f"Using elements: {_items[0]['name']!r} → {_items[1]['name']!r}")

✓ Backend available at http://localhost:8002
Using elements: 'schemaKey' → 'name'


## 1. Create an Element Mapping

An element mapping declares a relationship between two elements from potentially
different sources. The `function_type` field specifies how values should be
transformed — `identity` means no transformation (direct equivalence).

In [2]:
response = httpx.post(
    f"{BACKEND_URL}/api/v1/mappings/",
    headers=HEADERS,
    json={
        "function_type": "identity",
        "output_element_id": elem_b_id,
        "input_element_ids": [{"element_id": elem_a_id, "position": 0}],
        "description": "Tutorial identity mapping",
    },
    timeout=5.0,
)
if response.status_code == 409:
    # Already mapped in a previous run — retrieve existing mapping
    existing = httpx.get(
        f"{BACKEND_URL}/api/v1/mappings/",
        headers=HEADERS,
        params={"limit": 100},
        timeout=5.0,
    )
    mappings = existing.json().get("items", [])
    mapping = next(
        (m for m in mappings if m.get("output_element_id") == elem_b_id),
        None,
    )
    assert mapping is not None, "Could not find existing mapping after 409"
    mapping_id = mapping["id"]
    print(f"Reusing existing mapping: {mapping_id}")
else:
    assert response.status_code in (200, 201), (
        f"Expected 201, got {response.status_code}: {response.text}"
    )
    mapping = response.json()
    mapping_id = mapping["id"]
    print(f"Created mapping: {mapping_id}")
print(f"  function_type:     {mapping.get('function_type')}")
print(f"  output_element_id: {mapping.get('output_element_id')}")

Created mapping: 43e46269-531c-4b33-8a39-897e0ac4e580
  function_type:     identity
  output_element_id: 6516afac-dece-49df-bbbf-a8207101b424


## 2. Inspect the Mapping

Verify the mapping was persisted and check its version history.

In [3]:
response = httpx.get(
    f"{BACKEND_URL}/api/v1/mappings/{mapping_id}",
    headers=HEADERS,
    timeout=5.0,
)
assert response.status_code == 200, f"Expected 200, got {response.status_code}"
m = response.json()
print(f"Mapping {mapping_id}:")
print(f"  output_element_id: {m.get('output_element_id')}")
print(f"  function_type:     {m.get('function_type')}")
inputs = m.get("inputs", [])
print(f"  inputs:            {[i.get('element_id') for i in inputs]}")

hist_resp = httpx.get(
    f"{BACKEND_URL}/api/v1/mappings/{mapping_id}/history",
    headers=HEADERS,
    timeout=5.0,
)
if hist_resp.status_code == 200:
    hist = hist_resp.json()
    versions = hist if isinstance(hist, list) else hist.get("items", [])
    print(f"  history versions:  {len(versions)}")

Mapping 43e46269-531c-4b33-8a39-897e0ac4e580:
  output_element_id: 6516afac-dece-49df-bbbf-a8207101b424
  function_type:     identity
  inputs:            ['eafbbce8-0692-4114-855b-f69e85569f81']
  history versions:  1


## 3. Create an Alias Group

An alias group collects elements that are semantically equivalent across sources.
The `predicate` field uses SKOS vocabulary to specify the relationship type.

In [4]:
response = httpx.post(
    f"{BACKEND_URL}/api/v1/aliases/",
    headers=HEADERS,
    json={
        "name": "tutorial-alias-group",
        "sssom_predicate": "skos:exactMatch",
        "element_ids": [elem_a_id],
    },
    timeout=5.0,
)
if response.status_code == 409:
    # Already created in a previous run — retrieve it
    existing = httpx.get(
        f"{BACKEND_URL}/api/v1/aliases/",
        headers=HEADERS,
        params={"limit": 100},
        timeout=5.0,
    )
    groups = existing.json().get("items", [])
    alias_group = next((g for g in groups if g.get("name") == "tutorial-alias-group"), None)
    assert alias_group is not None, "Could not find existing alias group after 409"
    alias_group_id = alias_group["id"]
    print(f"Reusing existing alias group: {alias_group_id}")
else:
    assert response.status_code in (200, 201), (
        f"Expected 201, got {response.status_code}: {response.text}"
    )
    alias_group = response.json()
    alias_group_id = alias_group["id"]
    print(f"Created alias group: {alias_group_id}")
print(f"  name:            {alias_group.get('name')}")
print(f"  sssom_predicate: {alias_group.get('sssom_predicate')}")
print("✓ Alias group ready")

Created alias group: 8dc9a04c-9054-4dad-8c00-2d8008892f44
  name:            tutorial-alias-group
  sssom_predicate: skos:exactMatch
✓ Alias group ready


## 4. Run detect-aliases CLI

The `undata detect-aliases` command scans all elements for similarity above a
threshold and can automatically create alias groups. The `--dry-run` flag shows
what would be created without actually writing anything.

In [5]:
result = subprocess.run(
    [
        "uv",
        "run",
        "undata",
        "detect-aliases",
        "--dry-run",
        "--source-filter",
        "DANDI",
        "--backend-url",
        f"{BACKEND_URL}/api/v1",
        "--token",
        API_KEY,
    ],
    cwd=INGESTION_DIR,
    capture_output=True,
    text=True,
)
print("STDOUT:", result.stdout[-1000:] if result.stdout else "(none)")
if result.stderr:
    print("STDERR:", result.stderr[-500:])
assert result.returncode == 0, f"detect-aliases failed with code {result.returncode}"
print("✓ detect-aliases --dry-run complete")

STDOUT: xactMatch]
  EXACT  632ac641-139c-4604-b9b8-35aa6159ccbb ↔ 0d44299f-2221-4a64-89d4-a6cc6b5c1597  [score=0.93, skos:exactMatch]
  EXACT  632ac641-139c-4604-b9b8-35aa6159ccbb ↔ 8b155524-3ddc-4192-9388-72ccb6ea85a6  [score=0.93, skos:exactMatch]
  EXACT  c82b6626-0a09-4c0c-917c-f8911febba58 ↔ acf77efb-f0d8-4781-84ce-e169ac23ffb5  [score=0.93, skos:exactMatch]
  EXACT  c82b6626-0a09-4c0c-917c-f8911febba58 ↔ 2c85af24-eb7d-437a-b28e-21ef8211d99b  [score=0.93, skos:exactMatch]
  EXACT  0d44299f-2221-4a64-89d4-a6cc6b5c1597 ↔ acf77efb-f0d8-4781-84ce-e169ac23ffb5  [score=0.93, skos:exactMatch]
  EXACT  0d44299f-2221-4a64-89d4-a6cc6b5c1597 ↔ 2c85af24-eb7d-437a-b28e-21ef8211d99b  [score=0.93, skos:exactMatch]
  EXACT  acf77efb-f0d8-4781-84ce-e169ac23ffb5 ↔ 8b155524-3ddc-4192-9388-72ccb6ea85a6  [score=0.93, skos:exactMatch]
  EXACT  2c85af24-eb7d-437a-b28e-21ef8211d99b ↔ 8b155524-3ddc-4192-9388-72ccb6ea85a6  [score=0.93, skos:exactMatch]
Total: 4777 exact matches, 0 close matches candidates

## Cleanup

Remove the resources created by this tutorial.

In [6]:
# Delete mapping
del_m = httpx.delete(
    f"{BACKEND_URL}/api/v1/mappings/{mapping_id}",
    headers=HEADERS,
    timeout=5.0,
)
print(f"Deleted mapping {mapping_id}: {del_m.status_code}")

# Delete alias group
del_a = httpx.delete(
    f"{BACKEND_URL}/api/v1/aliases/{alias_group_id}",
    headers=HEADERS,
    timeout=5.0,
)
print(f"Deleted alias group {alias_group_id}: {del_a.status_code}")
print("✓ Cleanup complete")

Deleted mapping 43e46269-531c-4b33-8a39-897e0ac4e580: 200
Deleted alias group 8dc9a04c-9054-4dad-8c00-2d8008892f44: 204
✓ Cleanup complete


## Next Steps

You've created an element mapping, an alias group, and run the detect-aliases CLI.

Next: **[T05: LinkML Schema Export](05_linkml_export.ipynb)** — generate a unified
LinkML YAML schema from all backend elements using the `undata generate-schema` command.